# Milestone 4 - Incident Type Analysis & Dashboard

## Objective
The goal of this milestone is to perform a detailed analysis of disaster incident types and integrate insights from previous milestones (temporal and geographical analysis) into a coherent final narrative.

We will explore:
- Which disaster types occur most frequently?
- Which states experience specific disaster types the most?
- How do disaster assistance programs relate to different incident types?

## 1. Dataset Preparation

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('../data/processed/usnd_cleaned.csv')

df['declarationDate'] = pd.to_datetime(df['declarationDate'])

df.head()

,state,incidentType,declarationDate,year,month,ihProgramDeclared,paProgramDeclared
0,GA,Tornado,1953-05-02,1953,5,No,Yes
1,TX,Tornado,1953-05-15,1953,5,No,Yes
2,LA,Flood,1953-05-29,1953,5,No,Yes
3,MI,Tornado,1953-06-02,1953,6,No,Yes
4,MT,Flood,1953-06-06,1953,6,No,Yes


## 2. Incident Type Distribution Analysis
Understand the overall distribution of disaster types.

In [ ]:
incident_counts = df['incidentType'].value_counts().reset_index()
incident_counts.columns = ['Incident Type', 'Count']

fig_bar = px.bar(incident_counts, 
                 x='Incident Type', 
                 y='Count', 
                 title='Most Frequent Disaster Types',
                 color='Count',
                 color_continuous_scale='Reds')
fig_bar.show()

fig_tree = px.treemap(incident_counts, 
                      path=['Incident Type'], 
                      values='Count', 
                      title='Disaster Composition (Treemap)')
fig_tree.show()

## 3. State vs Incident Type Analysis
Analyze how disaster types are distributed across different states.

In [ ]:
state_incident_df = df.groupby(['state', 'incidentType']).size().reset_index(name='Count')

fig_stacked = px.bar(state_incident_df,
                     x='state',
                     y='Count',
                     color='incidentType',
                     title='Incident Type Contribution within Each State',
                     barmode='stack')
fig_stacked.update_layout(xaxis={'categoryorder':'total descending'})
fig_stacked.show()

heatmap_data = state_incident_df.pivot(index='state', columns='incidentType', values='Count').fillna(0)
fig_heat = px.imshow(heatmap_data,
                     labels=dict(x="Incident Type", y="State", color="Count"),
                     title='State vs Incident Type Heatmap',
                     aspect='auto',
                     color_continuous_scale='Viridis')
fig_heat.show()

## 4. Disaster Assistance Program Analysis
Analyze relationship between incidentType and assistance programs.

In [ ]:
df['IH_numeric'] = df['ihProgramDeclared'].map({'Yes': 1, 'No': 0})
df['PA_numeric'] = df['paProgramDeclared'].map({'Yes': 1, 'No': 0})

assistance_df = df.groupby('incidentType')[['IH_numeric', 'PA_numeric']].sum().reset_index()
assistance_df = assistance_df.melt(id_vars='incidentType', 
                                   value_vars=['IH_numeric', 'PA_numeric'],
                                   var_name='Program',
                                   value_name='Count')
assistance_df['Program'] = assistance_df['Program'].map({'IH_numeric': 'Individual Housing (IH)', 'PA_numeric': 'Public Assistance (PA)'})

fig_assistance = px.bar(assistance_df,
                        x='incidentType',
                        y='Count',
                        color='Program',
                        title='Disaster Assistance Programs by Incident Type',
                        barmode='group')
fig_assistance.show()